# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Dataset ID (@id): {metadata.id}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")
print(f"Number of record sets: {len(metadata.record_sets)}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List record sets and their @id
record_sets = metadata.record_sets
print("Available RecordSets:")
for rs in record_sets:
    print(f"  RecordSet name: {rs.name} | @id: {rs.id} | {len(rs.fields)} fields")

# List all fields in each RecordSet with their @id
for rs in record_sets:
    print(f"\nFields in RecordSet '{rs.name}' (@id: {rs.id}):")
    for field in rs.fields:
        print(f"  Field name: {field.name} | @id: {field.id} | dataType: {field.data_type}")

# If any RecordSet has columns (for tabular data), list columns
for rs in record_sets:
    if hasattr(rs, 'columns') and rs.columns:
        print(f"\nColumns in RecordSet '{rs.name}' (@id: {rs.id}):")
        for col in rs.columns:
            print(f"  Column name: {col.name} | @id: {col.id} | dataType: {col.data_type}")

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from all record sets
dataframes = {}
for rs in record_sets:
    record_set_id = rs.id
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df

# Display column names for each DataFrame
for record_set_id, df in dataframes.items():
    print(f"\nDataFrame columns for RecordSet @id: {record_set_id}")
    print(df.columns.tolist())
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example: Select a numeric field for analysis from the main record set
# Let's use the first available DataFrame
if len(dataframes) == 0:
    print("No tabular record sets found for EDA.")
else:
    # Pick the first RecordSet with data
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]

    # Attempt to identify numeric fields based on DataFrame types
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        threshold = df[numeric_field_id].mean()  # Use mean as threshold for illustration
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Try to group by a categorical field (e.g. anatomical location or MSI status)
        group_fields = [col for col in df.columns if pd.api.types.is_string_dtype(df[col])]
        if group_fields:
            group_field_id = group_fields[0]
            grouped_df = filtered_df.groupby(group_field_id).mean()
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
    else:
        print("No numeric fields found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only run if a DataFrame is available w/ numeric fields
if len(dataframes) > 0 and numeric_fields:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id} in RecordSet @id: {record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Scatter plot for two numeric fields if available
    if len(numeric_fields) > 1:
        plt.figure(figsize=(8, 6))
        sns.scatterplot(data=df, x=numeric_fields[0], y=numeric_fields[1])
        plt.title(f"Scatter plot of {numeric_fields[0]} vs {numeric_fields[1]}")
        plt.xlabel(numeric_fields[0])
        plt.ylabel(numeric_fields[1])
        plt.show()
else:
    print("No numeric fields available for visualization.")

## 6. Conclusion
This notebook demonstrated how to load and explore the FAIR^2 colorectal cancer dataset using the Croissant schema and `mlcroissant` library.

- Loaded rich metadata and listed available record sets, fields, and their `@id`s.
- Extracted tabular data for analysis and visualized numeric distributions.
- Applied filtering, normalization, and grouping to support clinical EDA.

Further exploration can be done by referencing field and column `@id`s for advanced analytics and modeling, ensuring reproducibility and traceability.